In [ ]:
from flask import Flask, request, jsonify
import mysql.connector
import requests
import uuid
import threading
from collections import Counter

app_order = Flask("OrderService")

# Map Distributors to their URLs (Ports 5001, 5002, 5003)
DISTRIBUTOR_URLS = {
    "TechWorld": "http://127.0.0.1:5001/order",
    "ElectroCom": "http://127.0.0.1:5002/order",
    "GadgetCentral": "http://127.0.0.1:5003/order"
}

def get_db():
    return mysql.connector.connect(host="localhost", user="root", password="", database="gadgethub_db")

@app_order.route('/place_order', methods=['POST'])
def place_order():
    data = request.json
    customer_name = data['customer']
    customer_email = data.get('email', 'no-email@example.com')
    items_requested = data['items'] 
    
    # 1. Get Quotes
    try:
        quote_res = requests.post("http://127.0.0.1:5004/get_best_quotes", json={"product_ids": items_requested})
        best_quotes = quote_res.json()
    except:
        return jsonify({"error": "Quotation Service Unavailable"}), 503

    total_amount = 0
    raw_order_items = [] 
    
    # 2. Select Best Distributor & Calculate Total
    for pid in items_requested:
        quote = best_quotes.get(pid)
        if quote and "error" not in quote:
            price = quote['price']
            distributor = quote['distributor']
            p_name = quote.get('product_name', 'Unknown Product') 
            total_amount += price
            raw_order_items.append((pid, p_name, price, distributor))
    
    if total_amount == 0:
        return jsonify({"error": "No items available"}), 400

    order_id = str(uuid.uuid4())
    conn = get_db()
    cursor = conn.cursor()
    
    try:
      
        # STEP A: Insert the Parent (Order Header) FIRST
        cursor.execute(
            "INSERT INTO orders (id, customer_name, customer_email, status, total_amount) VALUES (%s, %s, %s, %s, %s)",
            (order_id, customer_name, customer_email, "Confirmed", total_amount)
        )
        
        # STEP B: Now process items (Distributor calls + Child Inserts)
        grouped_items = Counter(raw_order_items)
        confirmed_details = []

        for (pid, p_name, price, distributor), qty in grouped_items.items():
            # 1. Call Distributor API to reduce stock
            dist_url = DISTRIBUTOR_URLS.get(distributor)
            if dist_url:
                try:
                    requests.post(dist_url, json={"product_id": pid, "quantity": qty})
                except:
                    print(f"Failed to contact {distributor}")

            # 2. Save Child Item to DB (Now safe because Order ID exists)
            cursor.execute(
                "INSERT INTO order_items (order_id, product_id, quantity, unit_price, distributor_name) VALUES (%s, %s, %s, %s, %s)",
                (order_id, pid, qty, price, distributor)
            )
            
            # Format for Receipt
            qty_str = f"{qty:02}"
            confirmed_details.append(f"{p_name} :Qty {qty_str} (from {distributor})")
            
        conn.commit()
        
        
    except Exception as e:
        conn.rollback()
        return jsonify({"error": str(e)}), 500
    finally:
        conn.close()

    # 4. Trigger Notification
    unique_product_names = list(set([i[1] for i in raw_order_items]))
    def call_notify():
        requests.post("http://localhost:5005/notifications/order-confirmation", json={
            "order_id": order_id,
            "customer_name": customer_name,
            "email": customer_email,
            "items": unique_product_names, 
            "total": total_amount,
            "details": confirmed_details 
        })
    threading.Thread(target=call_notify).start()

    # 5. Return Response
    final_receipt_lines = [f"{i+1}) {line}" for i, line in enumerate(confirmed_details)]
    
    return jsonify({
        "status": "Success",
        "order_id": order_id,
        "total": total_amount,
        "product_names": ", ".join(unique_product_names),
        "details": final_receipt_lines
    })

# RESTORED: GET Order Endpoint
@app_order.route('/orders/<order_id>', methods=['GET'])
def get_order(order_id):
    conn = get_db()
    cursor = conn.cursor(dictionary=True)
    cursor.execute("SELECT * FROM orders WHERE id=%s", (order_id,))
    order = cursor.fetchone()
    conn.close()
    if order:
        return jsonify(order)
    else:
        return jsonify({"error": "Order not found"}), 404

app_order.run(port=5000, use_reloader=False)

 * Serving Flask app 'OrderService'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [16/Feb/2026 14:13:03] "POST /place_order HTTP/1.1" 200 -
